# ЛР 3 - можно ли доверять LLM в задачах классического машинного обучения

Модели: GigaChat-3-Ultra, GPT-6.1 Sol (API) и Qwen3-4B-Instruct-2507 (локально на гугл колабе)

# Гигачат

GigaChat-3-Ultra. Ключ: `GIGACHAT_CREDENTIALS` в `.env`.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(".env");

from gigachat import GigaChat

giga_client = GigaChat(
    base_url="https://api.giga.chat/v1",
    credentials=os.environ["GIGACHAT_CREDENTIALS"],
    scope=os.getenv("GIGACHAT_SCOPE", "GIGACHAT_API_PERS"),
    model="GigaChat-3-Ultra",
    timeout=60,
)

def gigachat(prompt):
    response = giga_client.chat({
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 2048,
    })
    return response.choices[0].message.content

In [ ]:
prompt = "В двух предложениях объясни, что такое переобучение."
print(gigachat(prompt))

# ChatGPT

GPT-6.1 Sol через LLM Gate. Ключ: `LLM_GATE_API_KEY` в `.env`.

In [32]:
import os
from dotenv import load_dotenv

load_dotenv(".env");

from openai import OpenAI

gpt_client = OpenAI(
    base_url="https://api.llm-gate.space/v1",
    api_key=os.environ["LLM_GATE_API_KEY"],
    timeout=60,
    max_retries=0,
)

def chatgpt(prompt):
    response = gpt_client.chat.completions.create(
        model="gpt-6.1-sol",
        messages=[{"role": "user", "content": prompt}],
        reasoning_effort="low",
        max_completion_tokens=4096,
    )
    return response.choices[0].message.content

In [ ]:
prompt = "В двух предложениях объясни, что такое переобучение."
print(chatgpt(prompt))

# Общая функция и лог

`ask("gigachat" или "gpt", prompt, temperature)` возвращает ответ, общее число токенов и время в секундах. Успешные вызовы сохраняются в `answers.jsonl` рядом с тетрадью.

In [ ]:
import json
import time

SYSTEM_PROMPT = """Реши предложенную задачу. При необходимости приведи объяснение или код.
Если данных недостаточно для ответа, прямо укажи это.
Последнюю строку оформи строго так:
ОТВЕТ: <краткий итог> | УВЕРЕННОСТЬ: <целое число от 0 до 100>"""

def ask(model, prompt, temperature):
    start = time.perf_counter()
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt}]
    if model == "gigachat":
        response = giga_client.chat({
            "messages": messages, "temperature": temperature, "max_tokens": 2048})
    elif model == "gpt":
        response = gpt_client.chat.completions.create(
            model="gpt-6.1-sol", messages=messages, temperature=temperature,
            reasoning_effort="low", max_completion_tokens=4096)
    seconds = round(time.perf_counter() - start, 3)
    answer = response.choices[0].message.content
    tokens = response.usage.total_tokens
    record = {"model": model, "prompt": prompt, "temperature": temperature,
              "answer": answer, "tokens": tokens, "seconds": seconds}
    with open("answers.jsonl", "a", encoding="utf-8") as file:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")
    return answer, tokens, seconds

In [ ]:
answer, tokens, seconds = ask("gpt", "Что такое переобучение? Ответь кратко.", 0.2)
print(answer)
print(f"Токены: {tokens}, время: {seconds} с")

ОТВЕТ: 10 | УВЕРЕННОСТЬ: 100
Токены: 122, время: 2.152 с


# Отладка промпта

Пять задач из `test_bank.csv`, температура 0.

In [ ]:
import csv
import re

model = "gpt"  # gigachat

with open("test_bank.csv", encoding="utf-8-sig", newline="") as file:
    for task in csv.DictReader(file):
        answer, tokens, seconds = ask(model, task["текст_ru"], 0)
        last_line = answer.strip().splitlines()[-1]
        ok = bool(re.fullmatch(r"ОТВЕТ: .+ \| УВЕРЕННОСТЬ: (?:100|[1-9]?[0-9])", last_line))
        print(task["id"], "формат OK" if ok else "формат нарушен")
        print(answer, "\n")

T01 формат OK
Для решения этой задачи необходимо определить, что такое глубина бинарного дерева и как она связана с количеством листьев.

## Определение и анализ

*   **Бинарное дерево** — это структура данных, в которой каждый узел имеет не более двух потомков (левого и правого).
*   **Глубина узла** — это количество рёбер на пути от корня до этого узла. По условию, корень имеет глубину 0.
*   **Глубина дерева** — это максимальная глубина среди всех его узлов. В данном случае глубина дерева равна 4. Это означает, что самый длинный путь от корня до листа состоит из 4 рёбер, а значит, содержит 5 узлов (корень на глубине 0, узлы на глубинах 1, 2, 3 и лист на глубине 4).
*   **Лист** — это узел, у которого нет потомков.

Чтобы найти **максимальное** число листьев, нужно рассмотреть случай **идеально сбалансированного** (или полного) бинарного дерева. В таком дереве каждый нелистовой узел имеет ровно двух потомков.

На каждом уровне `d` (где `d` — это глубина) может находиться максимум $2^